# 공정 조합 퍼널 · 실제 데이터 분석

설비 이력 CSV(`raw.csv`)를 읽어서, 스텝 안의 오더별 유닛 조합 중 불량률을 끌어올린 **대표 불량 대상**을 찾고 랭킹과 차트로 보여 줍니다. 판정 로직은 main 브랜치의 웹 화면, `funnel_check.ipynb`와 같습니다.

**데이터 규칙**
- 웨이퍼 = `root_lot_id` + `wafer_id`. 웨이퍼마다 `y_value`가 하나입니다.
- 스텝 = `step_seq`, 오더 = `step_ord`, 유닛 = `eqp_id` + `-` + `chamber_id` (chamber가 없으면 `eqp_id`)

| 절 | 내용 |
|---|---|
| 0 | 설정 |
| 1 | 데이터 읽기와 점검: job · 제품 필터, 웨이퍼 키, 값 충돌, 결측, 재작업 중복 |
| 2 | 분석용 구조로 바꾸기: 스텝 · 오더 · 유닛 |
| 3 | 분석 · 차트 함수 (수정할 필요 없음) |
| 4 | 조합 판정과 랏 구조 점검 |
| 5 | 한 장의 funnel과 대표 불량 대상 랭킹 |
| 6 | 대표 불량 대상 자세히 보기: 후보, 스텝 흐름, 웨이퍼 분포, 왜 대표인가, 시간 추이 |
| 7 | 스텝 간 설비 연관 점검 |

## 0. 설정
데이터는 `unit_combi` 폴더(저장소 최상위)의 **`raw.csv` 하나만** 읽습니다. 저장소에는 가상 demo `raw.csv`가 들어 있으니, 실제 데이터로 덮어쓰고 실행하면 됩니다. 아래 값만 바꿔서 위에서부터 다시 실행하면 됩니다.

In [ ]:
JOB_ID = None         # 여러 job이 섞여 있으면 지정. None이면 analysis_date가 가장 최근인 job
PART_ID = None         # 보통 None: step_seq 기준으로 모든 part를 함께 분석. 한 제품만 볼 때만 지정
LINE_ID = None         # 보통 None. 한 라인만 볼 때만 지정
Y_COL = 'y_value'      # 판정에 쓸 값: 'y_value' 연속값(권장) · 'good_bad' bad 비율(정확 이항검정)
HIGHER_IS_WORSE = True # y_value가 클수록 나쁘면 True. 수율처럼 클수록 좋으면 False (부호를 뒤집어 분석)
PART_ADJUST = 'auto'   # part별 y_value 수준 차이 보정: 'auto' 차이가 뚜렷하면 보정 · True 항상 · False 안 함
REWORK = 'last'        # 같은 웨이퍼 · 스텝 · 오더 이력이 여러 줄이면: 'last' 마지막 track-in · 'first' 처음 track-in
MIN_N = 20             # 조합 최소 웨이퍼 수. 웨이퍼가 1,000장보다 적으면 10 정도로 낮춰 볼 것
MAX_DEPTH = 3          # 조합 최대 오더 수
ENV = 'auto'           # 판정 기준: 'auto' 랏 구조가 강하면 랏 단위 순열, 아니면 조합 수 보정 · 'bonf' · 'perm' · 'perm_lot'
N_PERM = 100           # 순열 기준을 쓸 때의 순열 횟수 (데이터가 크면 수 분)
RANK = 'impact'        # 랭킹 기준: 'impact' 영향 · 'evidence' 증거 · 'severity' 심각도
PICK = 1               # 6절에서 자세히 볼 대표 대상의 순위
STEP = None            # 특정 스텝만 볼 때 step_seq 값 (문자열). None이면 전체
FLOW_MAX_ORDERS = 12   # 스텝 흐름 차트에 그릴 최대 오더 수 (경로 · 유닛 간 차이가 있는 오더 우선)
LOGX = True            # funnel 가로축 로그
SAVE_RANKING = None    # 'ranking.csv'처럼 주면 랭킹 표를 CSV로 저장
FONT_PATH = None       # 따로 쓸 한글 글꼴(.ttf) 경로. None이면 노트북 옆 fonts 폴더의 나눔고딕
SEED = 0               # 순열 · 그림 흔들기용 난수 시드

In [ ]:
import math
import sys
import time
from collections import defaultdict
from itertools import combinations
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import to_rgb
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch, Patch, PathPatch, Rectangle
from matplotlib.path import Path as MplPath
from matplotlib.ticker import FuncFormatter, LogLocator, NullLocator

assert sys.version_info >= (3, 12), f'Python 3.12 이상이 필요합니다 (현재 {sys.version.split()[0]})'
assert Y_COL in ('y_value', 'good_bad'), "Y_COL은 'y_value' 또는 'good_bad'"
Y_NAME = 'bad 비율' if Y_COL == 'good_bad' else ('y_value' if HIGHER_IS_WORSE else '−y_value')

C = {
    'surface': '#fcfcfb', 'ink': '#0b0b0b', 'ink2': '#52514e', 'muted': '#898781',
    'grid': '#e1e0d9', 'axis': '#c3c2b7', 'band': '#ecebe6',
    'bad': '#d03b3b', 'accent': '#2a78d6', 'mid': '#f0efec',
}
# 한글 글꼴: FONT_PATH → 노트북 옆 fonts 폴더(저장소에 든 나눔고딕, SIL OFL) → 시스템 글꼴 순으로 찾는다
font_files = [Path(FONT_PATH)] if FONT_PATH else [
    p for d in (Path.cwd() / 'fonts', Path.cwd() / 'notebooks' / 'fonts') if d.is_dir() for p in sorted(d.glob('*.tt[fc]'))]
for p in font_files:
    mpl.font_manager.fontManager.addfont(str(p))
if font_files:
    KR_FONT = mpl.font_manager.FontProperties(fname=str(font_files[0])).get_name()
else:
    _fonts = {f.name for f in mpl.font_manager.fontManager.ttflist}
    KR_FONT = next((f for f in ('NanumGothic', 'Nanum Gothic', 'Noto Sans CJK KR', 'Noto Sans KR', 'Malgun Gothic',
                                'Apple SD Gothic Neo', 'AppleGothic') if f in _fonts), None)
if KR_FONT is None:
    print('⚠ 한글 글꼴을 찾지 못해 차트의 한글이 □로 보일 수 있습니다. 저장소의 notebooks/fonts 폴더를 노트북 옆에 두거나 FONT_PATH를 지정하세요.')
plt.rcParams.update({
    'font.family': [KR_FONT or 'DejaVu Sans', 'DejaVu Sans'], 'axes.unicode_minus': False, 'font.size': 10,
    'figure.dpi': 100, 'figure.facecolor': C['surface'], 'axes.facecolor': C['surface'], 'savefig.facecolor': C['surface'],
    'axes.edgecolor': C['axis'], 'axes.linewidth': 0.8, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': C['grid'], 'grid.linewidth': 0.8, 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left', 'axes.titlecolor': C['ink'], 'axes.titlepad': 10,
    'axes.labelcolor': C['ink2'], 'axes.labelsize': 10,
    'xtick.color': C['axis'], 'ytick.color': C['axis'], 'xtick.labelcolor': C['ink2'], 'ytick.labelcolor': C['ink2'],
    'xtick.major.size': 0, 'ytick.major.size': 0, 'xtick.minor.size': 0, 'ytick.minor.size': 0,
    'legend.frameon': False, 'legend.fontsize': 9,
})
pd.set_option('display.max_rows', 80)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', '{:,.4g}'.format)
print(f'Python {sys.version.split()[0]} · numpy {np.__version__} · pandas {pd.__version__} · matplotlib {mpl.__version__} · 글꼴 {KR_FONT}')

## 1. 데이터 읽기와 점검
- 컬럼을 확인하고, job · 제품 · 라인으로 좁힙니다. job이 여러 개면 가장 최근 분석을 씁니다.
- 웨이퍼마다 `y_value`, `good_bad`가 하나인지 확인합니다. 값이 둘 이상이면 첫 값을 쓰고 개수를 알립니다.
- 같은 웨이퍼 · 스텝 · 오더 이력이 여러 줄이면(재작업 등) `REWORK` 규칙으로 하나만 남깁니다.

In [ ]:
REQUIRED = ['analysis_date', 'job_id', 'root_lot_id', 'lot_id', 'wafer_id', 'tkin_time', 'line_id', 'part_id',
            'step_seq', 'step_desc', 'eqp_id', 'chamber_id', 'ppid', 'y_value', 'good_bad', 'step_ord']
BAD_TOKENS = {'b', 'bad', 'ng', 'n/g', 'fail', 'f', '1', 'true', 'y', 'yes', '불량'}
GOOD_TOKENS = {'g', 'good', 'ok', 'pass', 'p', '0', 'false', 'n', 'no', '양품'}

t0 = time.perf_counter()
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CSV_PATH = REPO / 'raw.csv'               # unit_combi 폴더의 raw.csv 하나만 읽는다
assert CSV_PATH.exists(), f'raw.csv가 없습니다: {CSV_PATH}'
print(f'읽는 파일: {CSV_PATH.resolve()}')
raw = pd.read_csv(CSV_PATH, dtype=str, low_memory=False)
miss_cols = [c for c in REQUIRED if c not in raw.columns]
assert not miss_cols, f'CSV에 없는 컬럼: {miss_cols}'
for c in REQUIRED:
    raw[c] = raw[c].str.strip()
n_read = len(raw)
print(f'읽기 {time.perf_counter() - t0:.1f}초 · {n_read:,}줄')

# 웨이퍼 키: wafer_id가 숫자면 '01'과 '1'을 같게 본다
num = pd.to_numeric(raw['wafer_id'], errors='coerce')
wid = raw['wafer_id'].copy()
is_int = num.notna() & (num == num.round())
wid[is_int] = num[is_int].astype(int).astype(str)
raw['wafer_key'] = raw['root_lot_id'] + '|' + wid

jobs = (raw.groupby('job_id', dropna=False)
        .agg(analysis_date=('analysis_date', 'max'), 줄=('job_id', 'size'), 웨이퍼=('wafer_key', 'nunique'))
        .sort_values('analysis_date'))
if len(jobs) > 1:
    if JOB_ID is None:
        JOB_ID = jobs.index[-1]
    print(f'⚠ job이 {len(jobs)}개 섞여 있어 {JOB_ID}만 씁니다 (JOB_ID로 바꿀 수 있음)')
    display(jobs)
for col, val in (('job_id', JOB_ID), ('part_id', PART_ID), ('line_id', LINE_ID)):
    if val is not None:
        raw = raw[raw[col] == val]
print(f"part_id {raw['part_id'].nunique()}개 · line_id {raw['line_id'].nunique()}개를 step_seq 기준으로 함께 분석합니다. "
      "part별 y_value 수준 차이는 4절에서 점검합니다.")

raw['y'] = pd.to_numeric(raw['y_value'], errors='coerce')
raw['ord'] = pd.to_numeric(raw['step_ord'], errors='coerce')
raw['t'] = pd.to_datetime(raw['tkin_time'], errors='coerce')
gb = raw['good_bad'].str.lower()
raw['bad'] = np.where(gb.isin(BAD_TOKENS), 1.0, np.where(gb.isin(GOOD_TOKENS), 0.0, np.nan))
unknown_gb = raw.loc[raw['good_bad'].notna() & raw['bad'].isna(), 'good_bad'].value_counts()

# 웨이퍼 표: 웨이퍼마다 값 하나
wf = raw.groupby('wafer_key').agg(root_lot_id=('root_lot_id', 'first'), part=('part_id', 'first'), part_n=('part_id', 'nunique'),
                                  y=('y', 'first'), y_n=('y', 'nunique'), bad=('bad', 'first'), bad_n=('bad', 'nunique'))
target_col = 'y' if Y_COL == 'y_value' else 'bad'
keep = wf[target_col].notna()

# 이력 표: 오더 · 설비가 있는 줄만, 유닛 = eqp_id-chamber_id
h = raw.loc[raw['wafer_key'].isin(wf.index[keep]) & raw['ord'].notna() & raw['eqp_id'].notna() & raw['step_seq'].notna(),
            ['wafer_key', 'step_seq', 'step_desc', 'ord', 'eqp_id', 'chamber_id', 't']].copy()
cham = h['chamber_id'].fillna('')
h['unit'] = np.where(cham.str.len() > 0, h['eqp_id'] + '-' + cham, h['eqp_id'])
n_dup_keys = int(h.duplicated(['wafer_key', 'step_seq', 'ord'], keep='first').sum())
h = (h.sort_values('t', na_position='first' if REWORK == 'last' else 'last')
      .drop_duplicates(['wafer_key', 'step_seq', 'ord'], keep=REWORK))

quality = pd.DataFrame([
    ('읽은 줄', n_read),
    ('필터 후 줄', len(raw)),
    ('웨이퍼 (root_lot_id + wafer_id)', len(wf)),
    (f'{Y_COL}가 없어 뺀 웨이퍼', int((~keep).sum())),
    ('y_value가 둘 이상인 웨이퍼 (첫 값 사용)', int((wf['y_n'] > 1).sum())),
    ('good_bad가 둘 이상인 웨이퍼 (첫 값 사용)', int((wf['bad_n'] > 1).sum())),
    ('part_id가 둘 이상인 웨이퍼 (첫 값 사용)', int((wf['part_n'] > 1).sum())),
    ('good_bad를 알 수 없는 줄', int(unknown_gb.sum())),
    ('step_ord · eqp_id · step_seq가 없어 뺀 줄', int((raw['ord'].isna() | raw['eqp_id'].isna() | raw['step_seq'].isna()).sum())),
    ('tkin_time을 읽지 못한 줄', int(raw['t'].isna().sum())),
    (f"같은 웨이퍼 · 스텝 · 오더가 겹친 줄 ('{REWORK}'만 남김)", n_dup_keys),
    ('분석에 쓰는 이력 줄', len(h)),
], columns=['항목', '값']).set_index('항목')
display(quality)
if len(unknown_gb):
    print('알 수 없는 good_bad 값:', dict(unknown_gb.head(10)))

## 2. 분석용 구조로 바꾸기
스텝마다 오더(`step_ord`)와 유닛 목록을 만들고, 웨이퍼마다 오더별로 어느 유닛을 지났는지 배열로 정리합니다. 유닛 이름에는 오더가 없어서, 경로는 `O2:EQP01-A → O3:EQP07-B`처럼 오더를 붙여 씁니다.

In [ ]:
MISSING = 0xFFFF


def step_sort_key(s):
    return (0, float(s), s) if s.replace('.', '', 1).isdigit() else (1, 0.0, s)


def ord_label(o):
    return str(int(o)) if float(o).is_integer() else str(o)


t0 = time.perf_counter()
wafers = wf.index[keep]
N = len(wafers)
widx = pd.Series(np.arange(N), index=wafers)
h['w'] = h['wafer_key'].map(widx).astype(int)
steps, assign, trackin = [], [], []
for s_i, (seq, g) in enumerate(sorted(h.groupby('step_seq'), key=lambda kv: step_sort_key(kv[0]))):
    desc = g['step_desc'].mode().iat[0] if g['step_desc'].notna().any() else ''
    ords = sorted(g['ord'].unique())
    A = np.full((len(ords), N), MISSING, dtype=np.uint16)
    seqs = []
    for q, o in enumerate(ords):
        go = g[g['ord'] == o]
        units = sorted(go['unit'].unique())
        assert len(units) < MISSING, f'{seq} 오더 {o}의 유닛이 너무 많습니다'
        A[q, go['w'].to_numpy()] = pd.Categorical(go['unit'], categories=units).codes
        seqs.append({'name': f'O{ord_label(o)}', 'units': units})
    tk = np.full(N, np.datetime64('NaT', 'ns'))
    first_t = g.groupby('w')['t'].min()
    tk[first_t.index.to_numpy()] = first_t.to_numpy()
    steps.append({'index': s_i, 'name': str(seq), 'proc': desc, 'seqs': seqs})
    assign.append(A)
    trackin.append(tk)

Y = wf.loc[wafers, target_col].to_numpy(float)
if Y_COL == 'y_value' and not HIGHER_IS_WORSE:
    Y = -Y                                     # 작을수록 나쁜 값은 부호를 뒤집어 '클수록 나쁨'으로 맞춘다
assert N >= 2 * MIN_N and Y.std() > 0, f'비교할 수 없습니다: 웨이퍼 {N}장, {Y_COL} 표준편차 {Y.std():.3g}'
bad_rate = np.nanmean(wf.loc[wafers, 'bad'].to_numpy(float)) if wf['bad'].notna().any() else math.nan
if bad_rate in (0.0, 1.0):
    print(f'⚠ good_bad가 한쪽뿐입니다 (bad 비율 {bad_rate:.0%}). 불량 웨이퍼만 들어 있다면 양품 웨이퍼도 함께 넣어야 비교할 수 있습니다.')
data = {'N': N, 'Y': Y, 'steps': steps, 'assign': assign, 'trackin': trackin, 'wafers': wafers,
        'lots': wf.loc[wafers, 'root_lot_id'].to_numpy(), 'parts': wf.loc[wafers, 'part'].fillna('').to_numpy(),
        'bad': wf.loc[wafers, 'bad'].to_numpy(float),
        'value': wf.loc[wafers, 'y'].to_numpy(float)}
n_orders = sum(len(s['seqs']) for s in steps)
print(f"변환 {time.perf_counter() - t0:.1f}초 · 웨이퍼 {N:,}장 · 랏 {len(set(data['lots'])):,}개 · 스텝 {len(steps)}개 · "
      f"오더 {n_orders}개 · 유닛 {sum(len(q['units']) for s in steps for q in s['seqs']):,}개")
print(f"{Y_NAME} 평균 {Y.mean():.4g} · 표준편차 {Y.std(ddof=1):.4g}"
      + (f" · bad 비율 {np.nanmean(data['bad']):.1%}" if np.isfinite(data['bad']).any() else ''))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.4))
cnt = np.bincount([len(s['seqs']) for s in steps])
axes[0].bar(np.arange(1, len(cnt)), cnt[1:], width=0.6, color=C['accent'])
axes[0].set(title='스텝별 오더 수', xlabel='오더 수', ylabel='스텝 수')
cnt = np.bincount([len(q['units']) for s in steps for q in s['seqs']])
axes[1].bar(np.arange(1, len(cnt)), cnt[1:], width=0.6, color=C['accent'])
axes[1].set(title='오더별 유닛 수', xlabel='유닛 수', ylabel='오더 수')
v, b = data['value'], data['bad']
if np.isfinite(b).any() and np.isfinite(v).any():
    bins = np.linspace(np.nanmin(v), np.nanmax(v), 50)
    axes[2].hist([v[b == 0], v[b == 1]], bins=bins, stacked=True, color=[C['muted'], C['bad']], label=['good', 'bad'],
                 edgecolor=C['surface'], linewidth=0.5)
    axes[2].legend()
else:
    axes[2].hist(v[np.isfinite(v)], bins=50, color=C['muted'])
axes[2].set(title='웨이퍼 y_value 분포', xlabel='y_value', ylabel='웨이퍼 수')
for ax in axes:
    ax.grid(axis='x', visible=False)
plt.tight_layout()
plt.show()

## 3. 분석 · 차트 함수
`funnel_check.ipynb`에서 웹 JS와 값 단위로 맞춰 본 로직을 그대로 옮겼습니다. 수정할 필요 없이 실행만 하면 됩니다.
- **조합**: 한 스텝 안에서 오더 1~`MAX_DEPTH`개의 유닛을 모두 지난 웨이퍼 묶음. 웨이퍼 `MIN_N`장 이상만 봅니다. 오더 3개 조합은 부모 조합이 z 1.96을 넘을 때만 만듭니다.
- **기준 z**: 오더 수별로 가능한 조합 수만큼 보정합니다(Bonferroni).
- **판정**: 기준을 넘고, 오더 2개 이상이면 오더 하나를 뺀 부모의 나머지 웨이퍼보다도 기준 z만큼 높아야(Welch t) 불량입니다. 더 구체적인 불량 조합으로 설명되면 '하위 기인'입니다.

In [ ]:
Z95, Z998, FWER_ALPHA = 1.959964, 3.090232, 0.05
SQRT2 = math.sqrt(2)
_A = (-39.69683028665376, 220.9460984245205, -275.9285104469687, 138.357751867269, -30.66479806614716, 2.506628277459239)
_B = (-54.47609879822406, 161.5858368580409, -155.6989798598866, 66.80131188771972, -13.28068155288572)
_C = (-0.007784894002430293, -0.3223964580411365, -2.400758277161838, -2.549732539343734, 4.374664141464968, 2.938163982698783)
_D = (0.007784695709041462, 0.3224671290700398, 2.445134137142996, 3.754408661907416)
_LG = (76.18009172947146, -86.50532032941677, 24.01409824083091, -1.231739572450155, 0.1208650973866179e-2, -0.5395239384953e-5)


def erfc(x):
    x = np.asarray(x, dtype=float)
    z = np.abs(x)
    t = 1 / (1 + 0.5 * z)
    poly = 1.00002368 + t * (0.37409196 + t * (0.09678418 + t * (-0.18628806 + t * (0.27886807 + t * (-1.13520398 + t * (1.48851587 + t * (-0.82215223 + t * 0.17087277)))))))
    r = t * np.exp(-z * z - 1.26551223 + t * poly)
    return np.where(x >= 0, r, 2 - r)


def norm_cdf(z):
    return 0.5 * erfc(-np.asarray(z, dtype=float) / SQRT2)


def norm_sf(z):
    return 0.5 * erfc(np.asarray(z, dtype=float) / SQRT2)


def norm_inv(p):
    if p <= 0:
        return -math.inf
    if p >= 1:
        return math.inf
    if p < 0.02425:
        q = math.sqrt(-2 * math.log(p))
        x = (((((_C[0] * q + _C[1]) * q + _C[2]) * q + _C[3]) * q + _C[4]) * q + _C[5]) / ((((_D[0] * q + _D[1]) * q + _D[2]) * q + _D[3]) * q + 1)
    elif p <= 1 - 0.02425:
        q = p - 0.5
        r = q * q
        x = ((((((_A[0] * r + _A[1]) * r + _A[2]) * r + _A[3]) * r + _A[4]) * r + _A[5]) * q) / (((((_B[0] * r + _B[1]) * r + _B[2]) * r + _B[3]) * r + _B[4]) * r + 1)
    else:
        q = math.sqrt(-2 * math.log(1 - p))
        x = -(((((_C[0] * q + _C[1]) * q + _C[2]) * q + _C[3]) * q + _C[4]) * q + _C[5]) / ((((_D[0] * q + _D[1]) * q + _D[2]) * q + _D[3]) * q + 1)
    e = float(norm_cdf(x)) - p
    u = e * math.sqrt(2 * math.pi) * math.exp((x * x) / 2)
    return x - u / (1 + (x * u) / 2)


def norm_isf(p):
    """상단 꼬리확률 → z (배열, 아주 작은 p도 정밀하게)"""
    p = np.clip(np.atleast_1d(np.asarray(p, dtype=float)), 1e-300, 1 - 1e-16)
    x = np.empty_like(p)
    lo, hi = p < 0.02425, p > 1 - 0.02425
    mid = ~(lo | hi)
    q = np.sqrt(-2 * np.log(p[lo]))
    x[lo] = (((((_C[0] * q + _C[1]) * q + _C[2]) * q + _C[3]) * q + _C[4]) * q + _C[5]) / ((((_D[0] * q + _D[1]) * q + _D[2]) * q + _D[3]) * q + 1)
    q = p[mid] - 0.5
    r = q * q
    x[mid] = ((((((_A[0] * r + _A[1]) * r + _A[2]) * r + _A[3]) * r + _A[4]) * r + _A[5]) * q) / (((((_B[0] * r + _B[1]) * r + _B[2]) * r + _B[3]) * r + _B[4]) * r + 1)
    q = np.sqrt(-2 * np.log(1 - p[hi]))
    x[hi] = -(((((_C[0] * q + _C[1]) * q + _C[2]) * q + _C[3]) * q + _C[4]) * q + _C[5]) / ((((_D[0] * q + _D[1]) * q + _D[2]) * q + _D[3]) * q + 1)
    with np.errstate(all='ignore'):
        e = norm_cdf(x) - p
        u = e * math.sqrt(2 * math.pi) * np.exp((x * x) / 2)
        refined = x - u / (1 + (x * u) / 2)
    return -np.where(np.isfinite(refined) & (p > 1e-250), refined, x)


def lgamma(x):
    x = np.asarray(x, dtype=float)
    y = x.copy()
    tmp = x + 5.5 - (x + 0.5) * np.log(x + 5.5)
    ser = 1.000000000190015
    for g in _LG:
        y = y + 1
        ser = ser + g / y
    return -tmp + np.log((2.5066282746310005 * ser) / x)


def betacf(a, b, x):
    a, b, x = np.broadcast_arrays(*(np.asarray(v, dtype=float) for v in (a, b, x)))
    MAXIT, EPS, FPMIN = 300, 3e-14, 1e-300
    qab, qap, qam = a + b, a + 1, a - 1
    floor = lambda v: np.where(np.abs(v) < FPMIN, FPMIN, v)
    c = np.ones_like(x)
    d = 1 / floor(1 - (qab * x) / qap)
    h_ = d.copy()
    live = np.ones(x.shape, dtype=bool)
    with np.errstate(all='ignore'):
        for m in range(1, MAXIT + 1):
            m2 = 2 * m
            aa = (m * (b - m) * x) / ((qam + m2) * (a + m2))
            d1 = 1 / floor(1 + aa * d)
            c1 = floor(1 + aa / c)
            h1 = h_ * (d1 * c1)
            aa = (-(a + m) * (qab + m) * x) / ((a + m2) * (qap + m2))
            d1 = 1 / floor(1 + aa * d1)
            c1 = floor(1 + aa / c1)
            de = d1 * c1
            h1 = h1 * de
            c, d, h_ = np.where(live, c1, c), np.where(live, d1, d), np.where(live, h1, h_)
            live &= ~(np.abs(de - 1) < EPS)
            if not live.any():
                break
    return h_


def ibeta(x, a, b):
    x, a, b = np.broadcast_arrays(*(np.atleast_1d(np.asarray(v, dtype=float)) for v in (x, a, b)))
    out = np.where(x <= 0, 0.0, 1.0)
    mid = (x > 0) & (x < 1)
    if mid.any():
        xm, am, bm = x[mid], a[mid], b[mid]
        bt = np.exp(lgamma(am + bm) - lgamma(am) - lgamma(bm) + am * np.log(xm) + bm * np.log(1 - xm))
        front = xm < (am + 1) / (am + bm + 2)
        r = np.empty_like(xm)
        if front.any():
            r[front] = (bt[front] * betacf(am[front], bm[front], xm[front])) / am[front]
        if (~front).any():
            r[~front] = 1 - (bt[~front] * betacf(bm[~front], am[~front], 1 - xm[~front])) / bm[~front]
        out[mid] = r
    return out


def t_sf(t, df):
    t, df = np.atleast_1d(np.asarray(t, dtype=float)), np.atleast_1d(np.asarray(df, dtype=float))
    p = 0.5 * ibeta(df / (df + t * t), df / 2, 0.5)
    return np.where(t >= 0, p, 1 - p)


def f_sf(f, d1, d2):
    f = np.atleast_1d(np.asarray(f, dtype=float))
    with np.errstate(all='ignore'):
        p = ibeta(d2 / (d2 + d1 * f), d2 / 2, d1 / 2)
    return np.where(f <= 0, 1.0, p)


def var_of(n, s, ss):
    return max(0.0, (ss - (s * s) / n) / (n - 1)) if n > 1 else 0.0


def anova(n, s, ss):
    keep_ = n > 0
    n, s, ss = n[keep_].tolist(), s[keep_].tolist(), ss[keep_].tolist()
    k, N_ = len(n), sum(n)
    if k < 2 or N_ - k < 1:
        return None
    grand = sum(s) / N_
    ssb = sum(ni * (si / ni - grand) ** 2 for ni, si in zip(n, s))
    ssw = sum(ssi - ni * (si / ni) ** 2 for ni, si, ssi in zip(n, s, ss))
    if ssw <= 0:
        return None
    d1, d2 = k - 1, N_ - k
    F = ssb / d1 / (ssw / d2)
    return {'F': F, 'd1': d1, 'd2': d2, 'p': float(f_sf(F, d1, d2)[0])}


def welch(na, sa, ssa, nb, sb, ssb):
    """a가 b보다 큰지 단측 Welch t (요약통계 배열)"""
    na, sa, ssa, nb, sb, ssb = (np.atleast_1d(np.asarray(v, dtype=float)) for v in (na, sa, ssa, nb, sb, ssb))
    t, df, p = np.zeros_like(na), np.ones_like(na), np.ones_like(na)
    ok = (na >= 2) & (nb >= 2)
    with np.errstate(all='ignore'):
        va = np.maximum(0, (ssa - (sa * sa) / na) / (na - 1)) / na
        vb = np.maximum(0, (ssb - (sb * sb) / nb) / (nb - 1)) / nb
        se = np.sqrt(va + vb)
        ok &= se != 0
        t[ok] = (sa[ok] / na[ok] - sb[ok] / nb[ok]) / se[ok]
        df[ok] = (va[ok] + vb[ok]) ** 2 / ((va[ok] * va[ok]) / (na[ok] - 1) + (vb[ok] * vb[ok]) / (nb[ok] - 1))
    if ok.any():
        p[ok] = t_sf(t[ok], df[ok])
    return t, df, p


def quantile(v, p):
    if not len(v):
        return math.nan
    h_ = (len(v) - 1) * p
    lo = math.floor(h_)
    return float(v[lo] + (h_ - lo) * (v[min(lo + 1, len(v) - 1)] - v[lo]))

In [ ]:
def combo_key(s, items):
    return f'{s}|' + '|'.join(f'{q}:{u}' for q, u in items)


def tuple_key(s, qs):
    return f'{s}|' + ','.join(map(str, qs))


def analyze(data, min_n=20, max_depth=3, full_depth=2, z_method='normal'):
    """조합 생성과 조합별 지표 (웹 js/analysis.js analyze와 같은 로직)"""
    N, Y, steps, assign = data['N'], data['Y'], data['steps'], data['assign']
    mu = float(Y.mean())
    sd = float(Y.std(ddof=1))
    Y2 = Y * Y
    if z_method == 'binom':
        def z_vec(n, s):
            n, x = np.asarray(n, dtype=float), np.rint(np.asarray(s, dtype=float))
            p = np.where(x <= 0, 1.0, ibeta(np.full_like(n, mu), np.maximum(x, 1), n - x + 1))
            return np.minimum(norm_isf(p), 38.0)
    else:
        def z_vec(n, s):
            n = np.asarray(n, dtype=float)
            return (np.asarray(s, dtype=float) / n - mu) / (sd / np.sqrt(n))

    groups = {}

    def cell_keys(s, qs):
        A = assign[s][list(qs)]
        dims = [len(steps[s]['seqs'][q]['units']) for q in qs]
        ok = np.all(A != MISSING, axis=0)
        key = np.zeros(N, dtype=np.int64)
        for i, d in enumerate(dims):
            key = key * d + A[i]
        return dims, ok, key

    def get_group(s, qs):
        ck = (s, tuple(qs))
        if (g := groups.get(ck)) is None:
            dims, ok, key = cell_keys(s, qs)
            size, k_ok = math.prod(dims), key[ok]
            g = {'qs': tuple(qs), 'dims': dims, 'n': np.bincount(k_ok, minlength=size),
                 'sum': np.bincount(k_ok, weights=Y[ok], minlength=size), 'ss': np.bincount(k_ok, weights=Y2[ok], minlength=size)}
            groups[ck] = g
        return g

    def count_cells(s, qs):
        dims, ok, key = cell_keys(s, qs)
        return int((np.bincount(key[ok], minlength=math.prod(dims)) >= min_n).sum())

    def decode(g, key):
        units = []
        for d in reversed(g['dims']):
            units.append(key % d)
            key //= d
        return tuple((q, int(u)) for q, u in zip(g['qs'], reversed(units)))

    def stats_of(s, items):
        g = get_group(s, [q for q, _ in items])
        key = 0
        for (_, u), d in zip(items, g['dims']):
            key = key * d + u
        return int(g['n'][key]), float(g['sum'][key]), float(g['ss'][key])

    seq_info = []
    for s, st in enumerate(steps):
        rows = []
        for q, sq in enumerate(st['seqs']):
            g = get_group(s, [q])
            units = [{'name': name, 'n': int(g['n'][u]), 'mean': g['sum'][u] / g['n'][u] if g['n'][u] else math.nan,
                      'sd': math.sqrt(var_of(int(g['n'][u]), g['sum'][u], g['ss'][u]))} for u, name in enumerate(sq['units'])]
            testable = len(sq['units']) >= 2
            rows.append({'name': sq['name'], 'units': units, 'testable': testable,
                         'anova': anova(g['n'], g['sum'], g['ss']) if testable else None})
        seq_info.append(rows)

    combos, by_key, flagged, flagged_tuples = [], {}, set(), defaultdict(set)

    def add(s, items, n, sm, ss, z):
        c = {'id': len(combos), 'key': combo_key(s, items), 'step': s, 'k': len(items), 'items': items,
             'n': int(n), 'sum': float(sm), 'ss': float(ss), 'mean': float(sm) / int(n), 'z': float(z)}
        combos.append(c)
        by_key[c['key']] = c
        if c['z'] > Z95:
            flagged.add(c['key'])
            flagged_tuples[c['k']].add(tuple_key(s, [q for q, _ in items]))

    multi = [[q for q, sq in enumerate(st['seqs']) if len(sq['units']) >= 2] for st in steps]
    search = [0] * (max_depth + 1)
    for k in range(1, max_depth + 1):
        gated = k > full_depth
        cand = []
        for s in range(len(steps)):
            if len(multi[s]) < k:
                continue
            for qs in combinations(multi[s], k):
                if gated and not any(tuple_key(s, qs[:i] + qs[i + 1:]) in flagged_tuples[k - 1] for i in range(k)):
                    search[k] += count_cells(s, qs)
                    continue
                g = get_group(s, qs)
                big = np.flatnonzero(g['n'] >= min_n)
                search[k] += len(big)
                cand.append((s, g, big))
        zs = z_vec(np.concatenate([g['n'][b] for _, g, b in cand]), np.concatenate([g['sum'][b] for _, g, b in cand])).tolist() if cand else []
        j = 0
        for s, g, big in cand:
            for key in big.tolist():
                z, j = zs[j], j + 1
                items = decode(g, key)
                if gated and not any(combo_key(s, items[:i] + items[i + 1:]) in flagged for i in range(k)):
                    continue
                add(s, items, g['n'][key], g['sum'][key], g['ss'][key], z)

    for c in combos:
        c['children'], c['parents'] = [], []
        c['resid'] = c['mean'] - mu if c['k'] == 1 else c['mean'] - (mu + sum(seq_info[c['step']][q]['units'][u]['mean'] - mu for q, u in c['items']))
    pairs = []
    for c in combos:
        for drop in range(c['k'] if c['k'] >= 2 else 0):
            p_items = c['items'][:drop] + c['items'][drop + 1:]
            pairs.append((c, p_items, *stats_of(c['step'], p_items)))
    if pairs:
        cn = np.array([p[0]['n'] for p in pairs], dtype=float)
        cs_ = np.array([p[0]['sum'] for p in pairs])
        css = np.array([p[0]['ss'] for p in pairs])
        pn, ps, pss = (np.array([p[i] for p in pairs], dtype=float) for i in (2, 3, 4))
        rn, rs, rss = pn - cn, ps - cs_, pss - css
        t, _, pv = welch(cn, cs_, css, rn, rs, rss)
        with np.errstate(all='ignore'):
            rmean = np.where(rn > 0, rs / rn, np.nan)
        for (c, p_items, n_, s_, _), rn_, rm_, t_, p_ in zip(pairs, rn.tolist(), rmean.tolist(), t.tolist(), pv.tolist()):
            c['parents'].append({'key': combo_key(c['step'], p_items), 'items': p_items, 'n': n_, 'mean': s_ / n_,
                                 'rest_n': int(rn_), 'rest_mean': rm_, 't': t_, 'p': p_})
    for c in combos:
        if c['k'] >= 2:
            c['lift_p'] = max(pr['p'] for pr in c['parents'])
            c['lift_t'] = min(pr['t'] for pr in c['parents'])
        else:
            c['lift_p'] = c['lift_t'] = None
        for pr in c['parents']:
            if (P := by_key.get(pr['key'])) is not None:
                P['children'].append(c['id'])
    depth_count = [0] * (max_depth + 1)
    for c in combos:
        depth_count[c['k']] += 1
    return {'mu': mu, 'sd': sd, 'N': N, 'max_depth': max_depth, 'z_method': z_method,
            'z_one': lambda n, s: float(z_vec(np.array([n]), np.array([s]))[0]),
            'seq_info': seq_info, 'combos': combos, 'by_key': by_key, 'depth_count': depth_count, 'search_space': search}


def thresholds(res, method='bonf', perm_z=None):
    z = [0.0]
    for k in range(1, res['max_depth'] + 1):
        if method == 'raw':
            z.append(Z998)
        elif method.startswith('perm') and perm_z and perm_z[k]:
            z.append(perm_z[k])
        else:
            z.append(norm_inv(1 - FWER_ALPHA / res['search_space'][k]) if res['search_space'][k] else Z998)
    return z


def classify(res, zk):
    """판정: 정상 · 경고 · 불량 · 상속 · 하위 기인 (웹 js/analysis.js classify와 같은 규칙)"""
    combos, z_one = res['combos'], res['z_one']
    for c in combos:
        c['over'] = c['z'] > zk[c['k']]
        c['explained_by'] = None
        if not c['over']:
            c['status'] = 'warn' if c['z'] > Z95 else 'normal'
        elif c['k'] == 1:
            c['status'] = 'bad'
        else:
            c['status'] = 'bad' if c['lift_t'] > zk[c['k']] else 'inherited'
    for k in range(res['max_depth'] - 1, 0, -1):
        for c in combos:
            if c['k'] != k or c['status'] != 'bad':
                continue
            for cid in c['children']:
                ch = combos[cid]
                if ch['status'] not in ('bad', 'explained'):
                    continue
                rn = c['n'] - ch['n']
                if rn >= 2 and z_one(rn, c['sum'] - ch['sum']) <= zk[k]:
                    c['status'], c['explained_by'] = 'explained', ch['id']
                    break
    count = lambda st: sum(c['status'] == st for c in combos)
    return {'over': sum(c['over'] for c in combos), 'bad': count('bad'), 'inherited': count('inherited'), 'explained': count('explained')}


def members_of(d, c):
    """조합의 웨이퍼 (불리언 배열)"""
    A = d['assign'][c['step']]
    m = np.ones(d['N'], dtype=bool)
    for q, u in c['items']:
        m &= A[q] == u
    return m


def perm_envelope(d, res, n_perm=100, alpha=FWER_ALPHA, seed=0, by_lot=False):
    """순열 기준선: Y를 섞어 '불량이 없을 때'의 깊이별 최대 z 분포를 만들고 (1−α) 분위수를 쓴다.
    by_lot이면 웨이퍼 수가 같은 랏끼리 랏을 통째로 바꿔 랏 안의 상관을 유지한다"""
    mems = [np.flatnonzero(members_of(d, c)) for c in res['combos']]
    lens = np.array([len(m) for m in mems])
    offsets = np.concatenate([[0], np.cumsum(lens)[:-1]])
    flat = np.concatenate(mems)
    depth = np.array([c['k'] for c in res['combos']])
    masks = {k: depth == k for k in range(1, res['max_depth'] + 1)}
    inv_se = np.sqrt(lens) / res['sd']
    rng = np.random.default_rng(seed)
    Yv = d['Y']
    if by_lot:
        lot_w = pd.Series(np.arange(d['N'])).groupby(d['lots']).apply(np.array)
        by_size = defaultdict(list)
        for ws in lot_w:
            by_size[len(ws)].append(ws)
        blocks = [np.stack(v) for v in by_size.values()]
    max_z = {k: [] for k in masks}
    for _ in range(n_perm):
        if by_lot:
            mp = np.empty(d['N'], dtype=np.int64)
            for bl in blocks:
                mp[bl.ravel()] = bl[rng.permutation(len(bl))].ravel()
            yp = Yv[mp]
        else:
            yp = Yv[rng.permutation(d['N'])]
        z = (np.add.reduceat(yp[flat], offsets) / lens - res['mu']) * inv_se
        for k, m in masks.items():
            if m.any():
                max_z[k].append(z[m].max())
    return [0.0] + [quantile(np.sort(max_z[k]), 1 - alpha) if max_z[k] else None for k in masks]

In [ ]:
RANK_COL = {'impact': '영향 (웨이퍼 × 차이)', 'evidence': 'z ÷ 기준 z', 'severity': '차이'}
STATUS_LABEL = {'bad': '불량', 'inherited': '관련 조합 · 상속', 'explained': '관련 조합 · 하위 기인', 'warn': '정상 범위', 'normal': '정상 범위'}
STATUS_SHORT = {'bad': '불량', 'inherited': '상속', 'explained': '하위 기인', 'warn': '기준 아래', 'normal': '기준 아래'}
in_scope = lambda c: STEP is None or steps[c['step']]['name'] == str(STEP)


def path_label(st, items, sep=' → '):
    return sep.join(f"{st['seqs'][q]['name']}:{st['seqs'][q]['units'][u]}" for q, u in items)


def metric(c, r, zk_, rank=RANK):
    match rank:
        case 'evidence':
            return c['z'] / zk_[c['k']]
        case 'severity':
            return c['mean'] - r['mu']
        case _:
            return c['n'] * (c['mean'] - r['mu'])


def candidates(r, c):
    """고른 조합과 같은 스텝에서 유닛을 하나 이상 공유하고 기준을 넘은 조합"""
    return [b for b in r['combos'] if b['over'] and b['step'] == c['step'] and set(b['items']) & set(c['items'])]


def representative(r, zk_, c, rank=RANK):
    bad = [b for b in candidates(r, c) if b['status'] == 'bad']
    return max(bad, key=lambda b: metric(b, r, zk_, rank)) if bad else None


def reason(r, zk_, st, b, target):
    if b['status'] == 'bad':
        return '대표 불량 대상' if b is target else '불량 (다른 대상)'
    if b['status'] == 'explained':
        return f"하위 기인: {path_label(st, r['combos'][b['explained_by']]['items'])} 때문"
    pr = min(b['parents'], key=lambda p: p['t'])
    return f"상속: {path_label(st, pr['items'])} 대비 t {pr['t']:.1f} ≤ 기준 {zk_[b['k']]:.1f}"


def combo_row(d, r, zk_, c):
    st = d['steps'][c['step']]
    m = members_of(d, c)
    row = {'스텝': st['name'], '설명': st['proc'], '경로': path_label(st, c['items']), '오더 수': c['k'], '웨이퍼': c['n'],
           '랏 수': len(np.unique(d['lots'][m])), f'평균 {Y_NAME}': c['mean'], '차이': c['mean'] - r['mu'],
           'z ÷ 기준 z': c['z'] / zk_[c['k']], '영향 (웨이퍼 × 차이)': c['n'] * (c['mean'] - r['mu'])}
    if np.isfinite(d['bad']).any():
        row['bad 비율'] = float(np.nanmean(d['bad'][m]))
    return row


def cross_step_check(d, r, zk_):
    """스텝 간 규칙: 다른 스텝의 불량 조합 A와 웨이퍼가 겹치는 불량 조합 B에 대해, A를 지나지 않은 웨이퍼만 놓고
    B가 나머지보다 높은지(Welch t) 본다. B의 효과가 사라지고(t ≤ 1.96) A는 B 없이도 기준을 넘으면 B를 'A 기인'으로 본다"""
    bads = [c for c in r['combos'] if c['status'] == 'bad']
    mem = {c['key']: members_of(d, c) for c in bads}
    y, y2 = d['Y'], d['Y'] ** 2

    def t_without(c, other):
        a, b = mem[c['key']] & ~mem[other['key']], ~mem[c['key']] & ~mem[other['key']]
        return float(welch(a.sum(), y[a].sum(), y2[a].sum(), b.sum(), y[b].sum(), y2[b].sum())[0][0])

    out = {}
    for b in bads:
        cause = [(t_without(b, a), a) for a in bads if a['step'] != b['step'] and (mem[a['key']] & mem[b['key']]).any()]
        cause = [(t, a) for t, a in cause if t <= Z95 and t_without(a, b) > zk_[a['k']]]
        if cause:
            out[b['key']] = min(cause, key=lambda ta: ta[0])[1]
    return out

In [ ]:
CHART_SEP = ' ⟶ '       # 차트 안 경로 구분자. 나눔고딕의 '→'는 아주 작게 찍혀서 차트에서는 긴 화살표를 쓴다 (표는 '→')


def mix(c1, c2, t):
    a, b = np.array(to_rgb(c1)), np.array(to_rgb(c2))
    return tuple(a * t + b * (1 - t))


def fmt_p(p):
    if p is None or not math.isfinite(p):
        return '–'
    return f'{p:.0e}' if p < 1e-4 else f'{p:.4f}'


def plain_log(axis):
    axis.set_major_locator(LogLocator(base=10, subs=(1, 2, 5)))
    axis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:,.0f}' if v >= 1 else f'{v:g}'))
    axis.set_minor_locator(NullLocator())


def label_points(ax, pts, gap=14):
    """불량 점 오른쪽에 이름. 점이 몰려 있으면 라벨을 무리의 오른쪽 바깥에 쌓고 가는 선으로 잇는다"""
    if not pts:
        return
    ax.autoscale_view()
    disp = ax.transData.transform([(x, y) for x, y, _ in pts])
    placed = []
    for i in np.argsort(disp[:, 1]):
        x0, y0 = disp[i]
        near = [j for j in range(len(pts)) if abs(disp[j][0] - x0) < 70 and abs(disp[j][1] - y0) < 20]
        lx, ly = max(disp[j][0] for j in near) + 9, y0
        for px, py in sorted(placed, key=lambda p: p[1]):
            if abs(lx - px) < 70 and abs(ly - py) < gap:
                ly = py + gap
        placed.append((lx, ly))
        x, y, text = pts[i]
        moved = len(near) > 1 or ly != y0
        ax.annotate(text, (x, y), xytext=(lx - x0 + (6 if moved else 0), ly - y0), textcoords='offset pixels', va='center',
                    fontsize=9, fontweight='bold', color=C['ink'], zorder=6,
                    arrowprops={'arrowstyle': '-', 'color': C['ink2'], 'lw': 0.8, 'shrinkA': 1, 'shrinkB': 5} if moved else None)


def plot_unified(ax, d, r, zk_, pick=None, cross=None, title=''):
    """모든 스텝 · 모든 조합 크기를 한 장의 funnel로. 세로축은 평균, 띠는 우연 범위 μ ± z₁·σ/√n.
    오더 2·3개 조합은 평균과의 차이를 z₁ ÷ z_k만큼 줄여 그려서 띠 하나로 모든 크기를 판정한다(띠 위 ⇔ z > z_k)"""
    cross = cross or {}
    cs = [c for c in r['combos'] if in_scope(c)]
    mu_, z1 = r['mu'], zk_[1]
    sc = lambda c: mu_ + (c['mean'] - mu_) * z1 / zk_[c['k']]
    fam = {b['key'] for b in candidates(r, pick)} if pick else set()
    target = representative(r, zk_, pick) if pick else None
    ns = [c['n'] for c in cs]
    lo, hi = max(2, min(ns) * 0.9), max(ns) * 1.1
    grid = lo * (hi / lo) ** (np.arange(80) / 79)
    half = z1 * r['sd'] / np.sqrt(grid)
    ax.fill_between(grid, mu_ - half, mu_ + half, color=C['band'], lw=0, zorder=0)
    for sign in (1, -1):
        ax.plot(grid, mu_ + sign * half, color=C['axis'], lw=1, zorder=1)
    ax.plot([lo, hi], [mu_, mu_], color=C['axis'], lw=1, ls=(0, (4, 3)), zorder=1)
    below = [c for c in cs if not c['over']]
    ax.scatter([c['n'] for c in below], [sc(c) for c in below], s=4, color=C['muted'], alpha=0.22, lw=0, zorder=1)
    for group, alpha in (([c for c in cs if c['over'] and c['key'] not in fam], 0.3 if pick else 1.0), ([c for c in cs if c['key'] in fam], 1.0)):
        bad = [c for c in group if c['status'] == 'bad' and c['key'] not in cross]
        rel = [c for c in group if c['status'] != 'bad']
        ax.scatter([c['n'] for c in rel], [sc(c) for c in rel], s=46, facecolors='none', edgecolors=C['bad'], lw=1.4, alpha=alpha, zorder=3)
        ax.scatter([c['n'] for c in bad], [sc(c) for c in bad], s=80, color=C['bad'], edgecolors=C['surface'], lw=1.8, alpha=alpha, zorder=4)
    xc = [c for c in cs if c['key'] in cross]
    ax.scatter([c['n'] for c in xc], [sc(c) for c in xc], s=110, facecolors=C['surface'], edgecolors=C['bad'], lw=2.4, zorder=5)
    if LOGX:
        ax.set_xscale('log')
        plain_log(ax.xaxis)
    steps_ = d['steps']
    if pick:
        ax.scatter([pick['n']], [sc(pick)], s=330, facecolors='none', edgecolors=C['accent'], lw=2.4, zorder=6)
        if target is not None and target is not pick:
            ax.annotate('', xy=(target['n'], sc(target)), xytext=(pick['n'], sc(pick)), zorder=5,
                        arrowprops={'arrowstyle': '-|>', 'color': C['accent'], 'lw': 1.6, 'shrinkA': 12, 'shrinkB': 9, 'connectionstyle': 'arc3,rad=-0.25'})
        ax.annotate('고른 점', (pick['n'], sc(pick)), xytext=(-12, -16), textcoords='offset points', ha='right', fontsize=9, color=C['ink2'])
        if target is not None:
            ax.annotate(f"대표 불량 대상\n{steps_[target['step']]['name']} · {path_label(steps_[target['step']], target['items'], CHART_SEP)}",
                        (target['n'], sc(target)), xytext=(12, 4), textcoords='offset points', va='center', fontsize=9, fontweight='bold', color=C['ink'])
    else:
        named = [(c['n'], sc(c), steps_[c['step']]['name']) for c in cs if c['status'] == 'bad' and c['key'] not in cross]
        if len(named) <= 12:
            label_points(ax, named)
    for c in xc:
        a = cross[c['key']]
        ax.annotate(f"{steps_[c['step']]['name']} · 다른 스텝 기인\n({steps_[a['step']]['name']} {path_label(steps_[a['step']], a['items'], CHART_SEP)})",
                    (c['n'], sc(c)), xytext=(12, -2), textcoords='offset points', va='center', fontsize=9, color=C['ink'])
    handles = [Line2D([], [], marker='o', ls='', color=C['bad'], mec=C['surface'], ms=8, label='불량'),
               Line2D([], [], marker='o', ls='', mfc='none', mec=C['bad'], ms=7, label='관련 조합'),
               Line2D([], [], marker='o', ls='', color=C['muted'], ms=4, label='우연 범위 안'),
               Patch(color=C['band'], label='우연 범위')]
    if pick:
        handles.append(Line2D([], [], marker='o', ls='', mfc='none', mec=C['accent'], mew=2, ms=11, label='고른 점'))
    if xc:
        handles.append(Line2D([], [], marker='o', ls='', mfc=C['surface'], mec=C['bad'], mew=2.2, ms=9, label='다른 스텝 기인'))
    ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(1.01, 1), borderaxespad=0)
    ax.set(title=title, xlabel='조합의 웨이퍼 수', ylabel=f'평균 {Y_NAME} (오더 2개 이상은 기준 환산)')
    ax.grid(axis='x', visible=False)


def plot_flow(c):
    """스텝 흐름: 오더 × 유닛 박스를 유닛 평균으로 색칠하고 경로를 잇는다"""
    st, info = steps[c['step']], res['seq_info'][c['step']]
    hit = dict(c['items'])
    qs_all = list(range(len(st['seqs'])))
    shown = qs_all
    if len(qs_all) > FLOW_MAX_ORDERS:                   # 오더가 많으면 경로 → 유닛 간 차이가 있는 오더 → 나머지 순으로 고른다
        sig = [q for q in qs_all if q not in hit and info[q]['anova'] and info[q]['anova']['p'] < 0.05]
        rest = sorted((q for q in qs_all if q not in hit and q not in sig), key=lambda q: (len(st['seqs'][q]['units']) < 2, q))
        shown = sorted((sorted(hit) + sig + rest)[:max(FLOW_MAX_ORDERS, len(hit))])
    pos = {q: i for i, q in enumerate(shown)}
    maxlen = max(len(u) for q in shown for u in st['seqs'][q]['units'])
    colW, gap, boxH, boxGap, top, padX = max(84, 7 * maxlen + 18), 26, 22, 4, 42, 2
    nq, maxU = len(shown), max(len(st['seqs'][q]['units']) for q in shown)
    W = padX * 2 + nq * colW + (nq - 1) * gap
    H = top + maxU * (boxH + boxGap) + 4
    max_dev = max([abs(res['sd']) * 0.1] + [u['mean'] - res['mu'] for sq in info for u in sq['units'] if not math.isnan(u['mean'])])
    col_x = lambda q: padX + pos[q] * (colW + gap)
    box_y = lambda u: top + u * (boxH + boxGap)
    fig = plt.figure(figsize=(W / 72, (H + 26) / 72))
    ax = fig.add_axes((0, 0, 1, H / (H + 26)))
    ax.set_xlim(0, W)
    ax.set_ylim(H, 0)
    ax.axis('off')
    note = f' · 오더 {len(qs_all)}개 중 {len(shown)}개 (경로와 유닛 간 차이가 있는 오더 우선)' if len(shown) < len(qs_all) else ''
    fig.text(0, 1, f"스텝 흐름 · {st['name']} {st['proc']} · 유닛 색 = 평균{note}", ha='left', va='top', fontsize=11, fontweight='bold', color=C['ink'])
    for q in shown:
        sq = st['seqs'][q]
        on, cx = q in hit, col_x(q) + colW / 2
        an_ = info[q]['anova']
        sub = '유닛 1개' if not info[q]['testable'] else (f"p {fmt_p(an_['p'])}" if an_ and an_['p'] < 0.05 else '차이 없음')
        ax.text(cx, 15, f"오더 {sq['name'][1:]}", ha='center', va='baseline', fontsize=12, fontweight='bold', color=C['bad'] if on else C['ink2'])
        ax.text(cx, 30, sub, ha='center', va='baseline', fontsize=10, color=C['muted'])
        for ui, u in enumerate(info[q]['units']):
            sel = on and hit[q] == ui
            t = 0.0 if math.isnan(u['mean']) else min(1.0, max(0.0, (u['mean'] - res['mu']) / max_dev))
            fill = C['bad'] if sel else mix(C['bad'], C['mid'], round(t * 55) / 100)
            alpha = 1 if on else 0.45
            ax.add_patch(FancyBboxPatch((col_x(q), box_y(ui)), colW, boxH, boxstyle='round,pad=0,rounding_size=6',
                                        fc=fill, ec='none', alpha=alpha, zorder=4 if sel else 3))
            ax.text(cx, box_y(ui) + boxH / 2 + 4, u['name'], ha='center', va='baseline', fontsize=10,
                    fontweight='bold' if sel else 'normal', color='white' if sel else C['ink'], alpha=alpha, zorder=5)
    pts = sorted((col_x(q) + colW / 2, box_y(u) + boxH / 2) for q, u in c['items'])
    if len(pts) > 1:
        verts, codes = [pts[0]], [MplPath.MOVETO]
        for (x0, y0), (x1, y1) in zip(pts, pts[1:]):
            dx = (x1 - x0) / 2
            verts += [(x0 + dx, y0), (x1 - dx, y1), (x1, y1)]
            codes += [MplPath.CURVE4] * 3
        ax.add_patch(PathPatch(MplPath(verts, codes), fc='none', ec=C['bad'], lw=3, alpha=0.85, capstyle='round', zorder=2))
    plt.show()


def membership(c):
    """이 경로 / 오더 하나만 다른 웨이퍼 / 이 경로가 아닌 나머지 (조합의 오더를 모두 지난 웨이퍼 기준)"""
    A = data['assign'][c['step']][[q for q, _ in c['items']]]
    U = np.array([u for _, u in c['items']])[:, None]
    present = np.all(A != MISSING, axis=0)
    mism = A != U
    miss = mism.sum(axis=0)
    me = np.flatnonzero(present & (miss == 0))
    others = np.flatnonzero(present & (miss > 0))
    parents = [np.flatnonzero(present & (miss == 1) & mism[i]) for i in range(c['k'])] if c['k'] >= 2 else []
    return me, others, parents


def box_stats(idx):
    v = np.sort(Y[idx])
    q1, q3 = quantile(v, 0.25), quantile(v, 0.75)
    iqr = q3 - q1
    return {'n': len(v), 'mean': float(v.mean()), 'q1': q1, 'med': quantile(v, 0.5), 'q3': q3,
            'lo': max(float(v[0]), q1 - 1.5 * iqr), 'hi': min(float(v[-1]), q3 + 1.5 * iqr), 'p01': quantile(v, 0.005), 'p99': quantile(v, 0.995)}


def plot_dist(ax, c, me, others, parents):
    st = steps[c['step']]
    groups = [('이 경로', me, C['bad'])]
    if c['k'] >= 2:
        for i, (q, u) in enumerate(c['items']):
            same = path_label(st, c['items'][:i] + c['items'][i + 1:], sep=' · ')
            groups.append((f"{same} 같음 · {st['seqs'][q]['name']} 다름", parents[i], C['ink2']))
    groups.append(('이 경로가 아닌 웨이퍼', others, C['muted']))
    groups = [g for g in groups if len(g[1])]
    sums = [box_stats(idx) for _, idx, _ in groups]
    rng, h_ = np.random.default_rng(SEED), 0.46
    for gi, ((label, idx, col), s_) in enumerate(zip(groups, sums)):
        pick_ = idx[::max(1, math.ceil(len(idx) / 500))]
        ax.scatter(Y[pick_], gi + (rng.random(len(pick_)) - 0.5) * 0.5, s=5, color=col, alpha=0.45 if gi == 0 else 0.2, lw=0, zorder=1)
        ax.plot([s_['lo'], s_['hi']], [gi, gi], color=col, lw=1.2, zorder=2)
        ax.add_patch(Rectangle((s_['q1'], gi - h_ / 2), s_['q3'] - s_['q1'], h_, fc=col, alpha=0.16, ec='none', zorder=2))
        ax.add_patch(Rectangle((s_['q1'], gi - h_ / 2), s_['q3'] - s_['q1'], h_, fc='none', ec=col, lw=1.4, zorder=3))
        ax.plot([s_['med']] * 2, [gi - h_ / 2, gi + h_ / 2], color=col, lw=2.5, zorder=3)
        ax.scatter([s_['mean']], [gi], marker='D', s=48, color=col, edgecolors=C['surface'], lw=1.5, zorder=4)
        ax.annotate(f"{s_['mean']:.3g} · {s_['n']:,}장", (s_['mean'], gi - h_ / 2), xytext=(0, 3), textcoords='offset points',
                    ha='center', va='bottom', fontsize=9, fontweight='bold', color=C['ink'], zorder=5)
    ax.axvline(res['mu'], color=C['muted'], ls=(0, (4, 3)), lw=1, zorder=0)
    ax.set_yticks(range(len(groups)), [g[0] for g in groups])
    ax.set_ylim(len(groups) - 0.5, -0.6)
    lo, hi = min(s_['p01'] for s_ in sums), max(s_['p99'] for s_ in sums)
    ax.set_xlim(lo - (hi - lo) * 0.03, hi + (hi - lo) * 0.03)
    ax.set_xlabel(Y_NAME)
    ax.set_title('웨이퍼 분포 · 상자 = 사분위 · ◆ = 평균 · 점선 = 전체 평균')
    ax.grid(axis='y', visible=False)


def wafer_stat(m):
    n = int(m.sum())
    if n < 2:
        return n, math.nan, math.nan
    v = Y[m]
    return n, float(v.mean()), 1.96 * float(v.std(ddof=1)) / math.sqrt(n)


G1 = '① 대표 불량 대상 ●'
G2 = '② 관련 조합 ● ⟶ 대표 대상의 웨이퍼를 빼면 ○'
G3 = '③ 대표 대상을 더 좁힌 조합 ● · 대표 대상의 나머지 웨이퍼 ○'


def why_rows(T, cap=6):
    st = steps[T['step']]
    mT, Ti = members_of(data, T), set(T['items'])
    lab = lambda items: path_label(st, items, CHART_SEP)
    if T['k'] == 1:
        (q, u), = T['items']
        A = data['assign'][T['step']][q]
        rows = [(G1, f"{lab(T['items'])} · ○ 같은 오더의 다른 유닛", wafer_stat(mT), wafer_stat((A != MISSING) & (A != u)))]
    else:
        rows = [(G1, lab(T['items']), wafer_stat(mT), None)]
    rel = [b for b in candidates(res, T) if b['key'] != T['key']]
    wider, seen = [], set()
    for i in range(T['k'] if T['k'] >= 2 else 0):
        items = T['items'][:i] + T['items'][i + 1:]
        key = combo_key(T['step'], items)
        wider.append(res['by_key'].get(key) or {'step': T['step'], 'items': items, 'status': 'normal', 'key': key})
        seen.add(key)
    wider += sorted((b for b in rel if b['key'] not in seen and not Ti < set(b['items'])), key=lambda b: -b['z'])[:cap]
    for c in wider:
        m = members_of(data, c)
        rows.append((G2, f"{lab(c['items'])} · {STATUS_SHORT[c['status']]}", wafer_stat(m), wafer_stat(m & ~mT)))
    for c in sorted((b for b in rel if Ti < set(b['items'])), key=lambda b: -b['z'])[:cap]:
        m = members_of(data, c)
        rows.append((G3, f"{lab(c['items'])} · {STATUS_SHORT[c['status']]}", wafer_stat(m), wafer_stat(mT & ~m)))
    return rows


def plot_why(T):
    rows = why_rows(T)
    layout = []
    for g in dict.fromkeys(r[0] for r in rows):
        layout.append(('header', g))
        layout += [('row', r) for r in rows if r[0] == g]
    fig, ax = plt.subplots(figsize=(12, 1.2 + 0.38 * len(layout)))
    for yy, (kind, item) in enumerate(layout):
        if kind == 'header':
            continue
        g, _, a, b = item
        col = C['bad'] if g == G1 else C['ink2']
        counts = f'● {a[0]:,}장'
        if b is not None and not math.isnan(b[1]):
            ax.plot([a[1], b[1]], [yy, yy], color=C['axis'], lw=2, zorder=1)
            ax.plot([b[1] - b[2], b[1] + b[2]], [yy, yy], color=C['muted'], lw=7, alpha=0.22, solid_capstyle='round', zorder=1)
            ax.scatter([b[1]], [yy], s=70, facecolors=C['surface'], edgecolors=C['ink2'], lw=1.6, zorder=3)
            counts += f' · ○ {b[0]:,}장'
        ax.plot([a[1] - a[2], a[1] + a[2]], [yy, yy], color=col, lw=7, alpha=0.2, solid_capstyle='round', zorder=1)
        ax.scatter([a[1]], [yy], s=80, color=col, edgecolors=C['surface'], lw=1.5, zorder=4)
        ax.annotate(counts, (1, yy), xycoords=('axes fraction', 'data'), xytext=(8, 0), textcoords='offset points',
                    va='center', fontsize=8, color=C['muted'], annotation_clip=False)
    ax.axvline(res['mu'], color=C['muted'], lw=1, ls=(0, (4, 3)), zorder=0)
    ax.axvline(T['mean'], color=C['bad'], lw=1, ls=(0, (1, 2)), zorder=0)
    ax.set_yticks(range(len(layout)), [item if kind == 'header' else item[1] for kind, item in layout])
    for lab, (kind, _) in zip(ax.get_yticklabels(), layout):
        if kind == 'header':
            lab.set_fontweight('bold')
            lab.set_color(C['ink'])
    ax.set_ylim(len(layout) - 0.4, -0.9)
    for x, text, colr in ((res['mu'], '전체 평균', C['ink2']), (T['mean'], '대표 대상 평균', C['bad'])):
        ax.annotate(text, (x, -0.9), xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9, color=colr, annotation_clip=False)
    ax.set_xlabel(f'평균 {Y_NAME} · 띠 = 95% 신뢰구간 · 오른쪽 숫자 = 웨이퍼 수')
    ax.set_title(f"왜 대표 불량 대상인가 · {steps[T['step']]['name']} {path_label(steps[T['step']], T['items'], CHART_SEP)}", pad=22)
    ax.grid(axis='y', visible=False)
    plt.tight_layout()
    plt.show()


def plot_time(T):
    """시간 추이: 이 스텝의 track-in 시각 순서로 이 경로와 스텝 전체의 이동평균"""
    s = T['step']
    t = data['trackin'][s]
    present = np.any(data['assign'][s] != MISSING, axis=0) & ~np.isnat(t)
    me = members_of(data, T) & ~np.isnat(t)
    if me.sum() < 3:
        print('track-in 시각이 있는 웨이퍼가 적어 시간 추이를 건너뜁니다.')
        return
    all_s = pd.Series(Y[present], index=pd.DatetimeIndex(t[present])).sort_index()
    me_s = pd.Series(Y[me], index=pd.DatetimeIndex(t[me])).sort_index()
    win_all = max(20, min(250, len(all_s) // 20))
    win_me = max(8, min(60, round(len(me_s) / 8)))
    fig, ax = plt.subplots(figsize=(13, 3.6))
    ax.scatter(me_s.index, me_s.values, s=6, color=C['bad'], alpha=0.25, lw=0, label='이 경로 웨이퍼')
    ax.plot(all_s.rolling(win_all).mean(), color=C['muted'], lw=2, label=f'스텝 전체 이동평균 ({win_all}장)')
    ax.plot(me_s.rolling(win_me).mean(), color=C['bad'], lw=2.5, label=f'이 경로 이동평균 ({win_me}장)')
    ax.axhline(res['mu'], color=C['muted'], lw=1, ls=(0, (4, 3)))
    rm = pd.concat([all_s.rolling(win_all).mean(), me_s.rolling(win_me).mean()]).dropna()
    if len(rm):
        pad = (rm.max() - rm.min()) * 0.3 + 1e-9
        ax.set_ylim(min(rm.min(), res['mu']) - pad, rm.max() + pad)
    ax.set(title=f"시간 추이 · {steps[s]['name']} track-in 시각", ylabel=Y_NAME)
    ax.legend(loc='lower right', bbox_to_anchor=(1, 1.0), ncol=3, borderaxespad=0.3)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

## 4. 조합 판정과 자동 점검
판정 전에 두 가지를 자동으로 점검하고, 필요하면 알아서 보정합니다. 설정을 따로 바꿀 필요는 없습니다.

**① part별 수준 차이**: 여러 part가 같은 `step_seq`를 지나므로 함께 분석합니다. 그런데 part마다 `y_value` 기준 수준이 다르고 특정 유닛이 특정 part에 몰리면, part 차이가 유닛 불량처럼 보입니다. part 간 차이가 뚜렷하면(`PART_ADJUST = 'auto'`) 웨이퍼 값에서 그 part의 평균을 빼고 전체 평균을 더해 수준을 맞춘 뒤 분석합니다.

**② 랏 구조**: 같은 랏의 웨이퍼들이 한 유닛에 함께 들어가고(랏 단위 배정), 랏마다 수준이 다르면(랏 간 편차), 유닛의 평균이 사실상 '랏 몇 개'로 정해집니다. 그러면 우연히 나빴던 랏 몇 개가 유닛 불량처럼 보입니다. 두 값이 모두 크면(`ENV = 'auto'`) 랏을 통째로 섞는 순열로 기준을 다시 잡습니다. 랭킹 표의 **랏 수**가 작은 대상(예: 3랏)은 이런 이유로 조심해서 봅니다.

In [ ]:
# ① part별 수준 차이 (y_value일 때만. 0/1인 good_bad는 보정하지 않음)
part_note = ''
if Y_COL == 'y_value' and len(set(data['parts'])) > 1 and PART_ADJUST is not False:
    pg = pd.DataFrame({'part': data['parts'], 'y': data['Y']}).groupby('part')['y']
    n_p, m_p = pg.size(), pg.mean()
    ss_b = float((n_p * (m_p - data['Y'].mean()) ** 2).sum())
    ss_t = float(((data['Y'] - data['Y'].mean()) ** 2).sum())
    an_p = anova(n_p.to_numpy(), pg.sum().to_numpy(), pg.apply(lambda v: (v * v).sum()).to_numpy())
    eta2, p_part = ss_b / ss_t, an_p['p'] if an_p else 1.0
    adjust = PART_ADJUST is True or (p_part < 0.001 and eta2 >= 0.01)
    display(pd.DataFrame({'웨이퍼': n_p, f'평균 {Y_NAME}': m_p}).sort_values('웨이퍼', ascending=False).head(10).rename_axis('part_id'))
    print(f'part 간 차이: 설명 비율 {eta2:.1%} · p {fmt_p(p_part)} → ' + ('part 평균을 맞춰 분석합니다' if adjust else '보정하지 않습니다'))
    if adjust:
        data['Y'] = data['Y'] - pg.transform('mean').to_numpy() + data['Y'].mean()
        Y = data['Y']
        Y_NAME += ' (part 보정)'
        part_note = ' · part 보정'

t0 = time.perf_counter()
res = analyze(data, min_n=MIN_N, max_depth=MAX_DEPTH, z_method='binom' if Y_COL == 'good_bad' else 'normal')
print(f"조합 {len(res['combos']):,}개 (오더 1/2/3개: {' / '.join(f'{v:,}' for v in res['depth_count'][1:])}) · "
      f"탐색 공간 {' / '.join(f'{v:,}' for v in res['search_space'][1:])} · {time.perf_counter() - t0:.1f}초")

# ② 랏 간 편차: y의 랏 간 분산 비율 (ICC, 일원분산분석 분산 성분)
lot_y = pd.DataFrame({'lot': data['lots'], 'y': Y}).groupby('lot')['y']
n_i = lot_y.size()
k_l, Nt = len(n_i), n_i.sum()
msb = (n_i * (lot_y.mean() - Y.mean()) ** 2).sum() / max(1, k_l - 1)
msw = (lot_y.var(ddof=1).fillna(0) * (n_i - 1)).sum() / max(1, Nt - k_l)
n0 = (Nt - (n_i ** 2).sum() / Nt) / max(1, k_l - 1)
icc = max(0.0, (msb - msw) / (msb + (n0 - 1) * msw)) if msb + (n0 - 1) * msw > 0 else 0.0
# 랏 단위 배정: (스텝, 오더)마다 웨이퍼 2장 이상인 랏 중 모든 웨이퍼가 한 유닛을 지난 랏의 비율
shares = []
for s, st in enumerate(steps):
    for q, sq in enumerate(st['seqs']):
        if len(sq['units']) < 2:
            continue
        a = assign[s][q]
        ok = a != MISSING
        per = pd.DataFrame({'lot': data['lots'][ok], 'u': a[ok]}).groupby('lot')['u'].agg(['nunique', 'size'])
        per = per[per['size'] >= 2]
        if len(per):
            shares.append((per['nunique'] == 1).mean())
lot_share = float(np.mean(shares)) if shares else 0.0
strong_lot = icc >= 0.05 and lot_share >= 0.5
print(f"랏 간 편차 비율(ICC) {icc:.1%} · 랏 단위 배정 비율 {lot_share:.0%} → " + ('랏 구조가 강함' if strong_lot else '랏 구조는 약함'))

ENV_DESC = {'bonf': '조합 수 보정', 'perm': '웨이퍼 순열', 'perm_lot': '랏 단위 순열'}
env = ('perm_lot' if strong_lot else 'bonf') if ENV == 'auto' else ENV
assert env in ENV_DESC, f"ENV는 'auto', 'bonf', 'perm', 'perm_lot' 중 하나: {ENV}"
envs = {'bonf': thresholds(res, 'bonf')}
if env != 'bonf':
    t0 = time.perf_counter()
    envs[env] = thresholds(res, env, perm_envelope(data, res, n_perm=N_PERM, seed=SEED, by_lot=env == 'perm_lot'))
    print(f'{ENV_DESC[env]} {N_PERM}회 {time.perf_counter() - t0:.0f}초')
display(pd.DataFrame({ENV_DESC[k]: v[1:] for k, v in envs.items()}, index=pd.Index(range(1, MAX_DEPTH + 1), name='오더 수')).rename_axis(columns='기준 z'))
zk = envs[env]
summary = classify(res, zk)
print(f"판정 기준: {ENV_DESC[env]}{part_note}" + (' (랏 구조가 강해 자동 선택)' if ENV == 'auto' and strong_lot else ''))
print(f"판정: 기준 밖 {summary['over']:,}개 → 불량 {summary['bad']}개 · 관련 조합(상속 {summary['inherited']}, 하위 기인 {summary['explained']})")

## 5. 한 장의 funnel과 대표 불량 대상 랭킹
모든 스텝 · 모든 조합 크기를 한 장에 그립니다. 우연 범위 밖의 점은 같은 스텝에서 유닛을 공유하는 **대표 불량 대상**으로 모아 랭킹을 매깁니다.
- **영향**: 웨이퍼 수 × 차이. 초과 불량의 총량, 조치하면 줄어드는 양
- **z ÷ 기준 z**: 우연이 아닐 확실성
- **차이**: 웨이퍼 한 장당 심각도. 웨이퍼가 적은 조합은 우연히 커질 수 있음
- **랏 수**: 조합의 웨이퍼가 속한 랏 수. 적으면 몇 개 랏의 우연일 수 있으니 조심해서 봄
- **스텝 간 판정**: 다른 스텝의 불량이 옮겨 온 것으로 보이면 '다른 스텝 기인'

In [ ]:
over_pts = [c for c in res['combos'] if c['over'] and in_scope(c)]
by_target = defaultdict(list)
for c in over_pts:
    t = representative(res, zk, c)
    by_target[t['key'] if t else None].append(c)
cross = cross_step_check(data, res, zk)
n_shown = sum(in_scope(c) for c in res['combos'])
print(f"우연 범위 밖의 점 {len(over_pts):,}개 → 대표 불량 대상 {sum(k is not None for k in by_target)}개"
      + (f" · 확정 불량 없음 {len(by_target[None])}개" if None in by_target else ''))
fig, ax = plt.subplots(figsize=(13, 5.2))
plot_unified(ax, data, res, zk, cross=cross, title=f'모든 스텝 · 모든 조합 크기 {n_shown:,}개 · 우연 범위 밖 {len(over_pts):,}개')
plt.tight_layout()
plt.show()

rep_rows = []
for key, pts in by_target.items():
    if key is not None:
        c = res['by_key'][key]
        a = cross.get(key)
        rep_rows.append({**combo_row(data, res, zk, c), '연결된 점': len(pts),
                         '스텝 간 판정': f"다른 스텝 기인 ({steps[a['step']]['name']} {path_label(steps[a['step']], a['items'])})" if a else '–',
                         'key': key})
rep_df = pd.DataFrame(rep_rows)
if len(rep_df):
    rep_df = rep_df.sort_values(RANK_COL[RANK], ascending=False)
    rep_df.index = pd.RangeIndex(1, len(rep_df) + 1, name=f'순위 ({RANK_COL[RANK]})')
    display(rep_df)
    if SAVE_RANKING:
        rep_df.to_csv(SAVE_RANKING, encoding='utf-8-sig')
        print(f'랭킹 표 저장: {SAVE_RANKING}')
else:
    print('대표 불량 대상이 없습니다.')

## 6. 대표 불량 대상 자세히 보기
랭킹 `PICK`위 대상을 봅니다.
- **후보 표**: 같은 스텝에서 기준을 넘은 관련 조합과, 각 후보가 대표가 아닌 이유
- **funnel**: 후보 중 기준을 겨우 넘은 관련 조합을 골랐을 때 대표 대상으로 이어지는 모습
- **스텝 흐름 · 웨이퍼 분포 · 왜 대표인가 · 시간 추이**

In [ ]:
if len(rep_df):
    T = res['by_key'][rep_df.iloc[PICK - 1]['key']]
    st = steps[T['step']]
    print(f"대표 불량 대상 {PICK}위: {st['name']} {st['proc']} · {path_label(st, T['items'])} · 웨이퍼 {T['n']:,}장 · "
          f"평균 {T['mean']:.4g} (전체 {res['mu']:.4g}, 차이 {T['mean'] - res['mu']:+.4g})")
    cands = sorted(candidates(res, T), key=lambda b: -metric(b, res, zk))
    display(pd.DataFrame([{**combo_row(data, res, zk, b), '판정': reason(res, zk, st, b, T)} for b in cands],
                         index=pd.RangeIndex(1, len(cands) + 1, name='후보')).drop(columns=['스텝', '설명']))
    rel = [b for b in cands if b['status'] != 'bad']
    pick = min(rel, key=lambda b: b['z'] / zk[b['k']]) if rel else T
    fig, ax = plt.subplots(figsize=(13, 5.2))
    plot_unified(ax, data, res, zk, pick=pick, title=f'고른 점 ⟶ 같은 스텝의 후보 {len(cands)}개 중 대표 불량 대상')
    plt.tight_layout()
    plt.show()

In [ ]:
if len(rep_df):
    plot_flow(T)
    me, others, parents = membership(T)
    fig, ax = plt.subplots(figsize=(12, 1.6 + 0.62 * (T['k'] + 2 if T['k'] >= 2 else 2)))
    plot_dist(ax, T, me, others, parents)
    plt.tight_layout()
    plt.show()

**왜 대표 불량 대상인가**: ●는 조합 전체, ○는 비교할 웨이퍼, 띠는 95% 신뢰구간입니다.
- **②** 관련 조합에서 대표 대상의 웨이퍼를 빼면(● → ○) 전체 평균으로 돌아갑니다. 관련 조합이 높았던 것은 대표 대상의 웨이퍼 때문입니다.
- **③** 대표 대상에 다른 오더의 유닛을 더해 좁힌 조합(●)이 대표 대상의 나머지 웨이퍼(○)와 같은 수준입니다. 더 좁혀도 더 나빠지지 않으므로, 더 구체적인 경로가 원인이 아닙니다.

In [ ]:
if len(rep_df):
    plot_why(T)
    plot_time(T)

## 7. 스텝 간 설비 연관 점검
판정은 스텝마다 따로 하므로, 두 스텝의 설비가 짝지어 운영되면 한 원인이 여러 스텝의 불량 조합으로 보일 수 있습니다. 불량 조합끼리 웨이퍼가 겹치면, 상대 스텝의 웨이퍼를 뺀 나머지로 효과가 남는지 봅니다.

In [ ]:
if cross:
    display(pd.DataFrame([{
        '불량 조합': f"{steps[res['by_key'][k]['step']]['name']} {path_label(steps[res['by_key'][k]['step']], res['by_key'][k]['items'])}",
        '원인으로 보이는 조합': f"{steps[a['step']]['name']} {path_label(steps[a['step']], a['items'])}",
    } for k, a in cross.items()], index=pd.RangeIndex(1, len(cross) + 1, name='다른 스텝 기인')))
else:
    print('다른 스텝의 불량이 옮겨 온 것으로 보이는 불량 조합이 없습니다.')